<a href="https://colab.research.google.com/github/henriquecrispim/interactive-performance-dashboard/blob/main/interactive_performance_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ==============================================================================
# DASHBOARD INTERATIVO DE PERFORMANCE
# ==============================================================================

# 1. Instalação e Importação de Dependências
!pip install plotly ipywidgets pandas numpy --quiet
from google.colab import output
output.enable_custom_widget_manager()
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from datetime import datetime, timedelta

# 2. Geração de Dados Sintéticos Estruturados (Simulação de Planilha de Treino)
def gerar_dados_treino(num_semanas=12):
    np.random.seed(42)
    datas = [datetime(2026, 1, 1) + timedelta(days=i) for i in range(num_semanas * 7)]

    tipos_treino = ['Rodagem (Z2)', 'Intervalado (Z4/Z5)', 'Tempo Run (Z3)', 'Longo (Z2)']
    probabilidades = [0.60, 0.15, 0.10, 0.15] # Distribuição 80/20

    dados = []
    for data in datas:
        if np.random.rand() > 0.43:
            tipo = np.random.choice(tipos_treino, p=probabilidades)

            if 'Longo' in tipo:
                distancia = round(np.random.uniform(14.0, 22.0), 2)
                pace_decimal = np.random.uniform(5.5, 6.2)
                fc_media = np.random.randint(130, 145)
            elif 'Intervalado' in tipo:
                distancia = round(np.random.uniform(6.0, 10.0), 2)
                pace_decimal = np.random.uniform(4.2, 4.8)
                fc_media = np.random.randint(165, 185)
            elif 'Tempo Run' in tipo:
                distancia = round(np.random.uniform(8.0, 12.0), 2)
                pace_decimal = np.random.uniform(4.8, 5.3)
                fc_media = np.random.randint(150, 165)
            else:
                distancia = round(np.random.uniform(6.0, 12.0), 2)
                pace_decimal = np.random.uniform(5.3, 5.8)
                fc_media = np.random.randint(135, 150)

            minutos = int(pace_decimal)
            segundos = int((pace_decimal - minutos) * 60)
            pace_formatado = f"{minutos}:{segundos:02d}"

            dados.append({
                "Data": data.strftime("%Y-%m-%d"),
                "Tipo_Treino": tipo,
                "Distancia_km": distancia,
                "Pace_Decimal": round(pace_decimal, 2),
                "Pace_Min_km": pace_formatado,
                "FC_Media_BPM": fc_media
            })

    return pd.DataFrame(dados)

df_treinos = gerar_dados_treino()

# 3. Função de Atualização Dinâmica do Dashboard (Engine Visual)
def atualizar_dashboard(tipo_filtro):
    if tipo_filtro == 'Todos os Treinos':
        df_filtrado = df_treinos
    else:
        df_filtrado = df_treinos[df_treinos['Tipo_Treino'] == tipo_filtro]

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.15,
        subplot_titles=("Evolução do Ritmo (Pace Médio por Km) ao Longo do Tempo",
                        "Relação entre Volume de Distância (Km) e Intensidade Cardíaca (BPM)")
    )

    # Gráfico 1: Linha de Evolução do Pace
    fig.add_trace(
        go.Scatter(
            x=df_filtrado['Data'], y=df_filtrado['Pace_Decimal'],
            mode='lines+markers',
            name='Pace (min/km)',
            line=dict(color='#d9534f', width=2),
            marker=dict(size=6),
            text=df_filtrado['Pace_Min_km'],
            hovertemplate="Data: %{x}<br>Ritmo: %{text} min/km<extra></extra>"
        ),
        row=1, col=1
    )

    # Gráfico 2: Barras de Distância
    fig.add_trace(
        go.Bar(
            x=df_filtrado['Data'], y=df_filtrado['Distancia_km'],
            name='Distância (Km)',
            marker_color='#2b5c8f',
            opacity=0.75,
            hovertemplate="Distância: %{y} km<extra></extra>"
        ),
        row=2, col=1
    )

    # Ajustes finos de Layout Estilizado
    fig.update_layout(
        template="plotly_dark",
        height=700,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=40, t=60, b=40)
    )

    # CORREÇÃO AQUI: Mudança de "reverse" para "reversed"
    fig.update_yaxes(title_text="Pace Decimal (Menor = Mais Rápido)", autorange="reversed", row=1, col=1)
    fig.update_yaxes(title_text="Volume de Distância (Km)", row=2, col=1)
    fig.update_xaxes(title_text="Linha do Tempo", row=2, col=1)

    fig.show()

# 4. Construção da Interface com Menu Suspenso (Dropdown Widgets)
opcoes_menu = ['Todos os Treinos'] + list(df_treinos['Tipo_Treino'].unique())
menu_dropdown = widgets.Dropdown(
    options=opcoes_menu,
    value='Todos os Treinos',
    description='Filtrar Treino:',
    disabled=False,
    style={'description_width': 'initial'}
)

print("[SUCESSO] Inicializando Dashboard de Performance Humana...")
widgets.interactive(atualizar_dashboard, tipo_filtro=menu_dropdown)

[SUCESSO] Inicializando Dashboard de Performance Humana...


interactive(children=(Dropdown(description='Filtrar Treino:', options=('Todos os Treinos', np.str_('Intervalad…

In [8]:
# Célula extra para rodar apenas se o erro persistir no GitHub
import json

# Carrega o próprio notebook (substitua pelo nome do seu arquivo se baixado)
try:
    with open('seu_notebook.ipynb', 'r', encoding='utf-8') as f:
        nb_data = json.load(f)

    # Remove a chave problemática de metadados se ela existir
    if 'widgets' in nb_data.get('metadata', {}):
        del nb_data['metadata']['widgets']

    with open('seu_notebook.ipynb', 'w', encoding='utf-8') as f:
        json.dump(nb_data, f, indent=2)
    print("Metadados corrigidos com sucesso!")
except:
    print("Para notebooks rodando direto na nuvem, use a Solução 1 (Limpar saídas).")

Para notebooks rodando direto na nuvem, use a Solução 1 (Limpar saídas).
